In [1]:
class Vector:
    def __init__(self, data):
        self.data = list(data)
        self.size = len(self.data)

    def __repr__(self):
        return f"Vector({self.data})"

    def __add__(self, other):
        return Vector([a + b for a, b in zip(self.data, other.data)])

    def __sub__(self, other):
        return Vector([a - b for a, b in zip(self.data, other.data)])

    def __mul__(self, scalar):
        return Vector([x * scalar for x in self.data])

    def dot(self, other):
        return sum(a * b for a, b in zip(self.data, other.data))

    def magnitude(self):
        return sum(x ** 2 for x in self.data) ** 0.5

In [2]:
class Matrix:
    def __init__(self, data):
        self.data = [list(row) for row in data]
        self.rows = len(self.data)
        self.cols = len(self.data[0])
        self.shape = (self.rows, self.cols)

    def __repr__(self):
        rows_str = "\n  ".join(str(row) for row in self.data)
        return f"Matrix({self.shape}):\n  {rows_str}"

    def __add__(self, other):
        return Matrix([
            [self.data[i][j] + other.data[i][j] for j in range(self.cols)]
            for i in range(self.rows)
        ])

    def __sub__(self, other):
        return Matrix([
            [self.data[i][j] - other.data[i][j] for j in range(self.cols)]
            for i in range(self.rows)
        ])

    def scalar_multiply(self, scalar):
        return Matrix([
            [self.data[i][j] * scalar for j in range(self.cols)]
            for i in range(self.rows)
        ])

    def element_wise_multiply(self, other):
        return Matrix([
            [self.data[i][j] * other.data[i][j] for j in range(self.cols)]
            for i in range(self.rows)
        ])

    def matmul(self, other):
        return Matrix([
            [
                sum(self.data[i][k] * other.data[k][j] for k in range(self.cols))
                for j in range(other.cols)
            ]
            for i in range(self.rows)
        ])

    def transpose(self):
        return Matrix([
            [self.data[j][i] for j in range(self.rows)]
            for i in range(self.cols)
        ])

    def determinant(self):
        if self.shape == (1, 1):
            return self.data[0][0]
        if self.shape == (2, 2):
            return self.data[0][0] * self.data[1][1] - self.data[0][1] * self.data[1][0]
        det = 0
        for j in range(self.cols):
            minor = Matrix([
                [self.data[i][k] for k in range(self.cols) if k != j]
                for i in range(1, self.rows)
            ])
            det += ((-1) ** j) * self.data[0][j] * minor.determinant()
        return det

    def inverse_2x2(self):
        det = self.determinant()
        if det == 0:
            raise ValueError("矩阵是奇异的，不存在逆矩阵")
        return Matrix([
            [self.data[1][1] / det, -self.data[0][1] / det],
            [-self.data[1][0] / det, self.data[0][0] / det]
        ])

    @staticmethod
    def identity(n):
        return Matrix([
            [1 if i == j else 0 for j in range(n)]
            for i in range(n)
        ])

In [3]:
A = Matrix([[1, 2], [3, 4]])
B = Matrix([[5, 6], [7, 8]])

print("A + B =", (A + B).data)
print("A @ B =", A.matmul(B).data)
print("A^T =", A.transpose().data)
print("det(A) =", A.determinant())
print("A^-1 =", A.inverse_2x2().data)

I = Matrix.identity(2)
print("A @ A^-1 =", A.matmul(A.inverse_2x2()).data)

A + B = [[6, 8], [10, 12]]
A @ B = [[19, 22], [43, 50]]
A^T = [[1, 3], [2, 4]]
det(A) = -2
A^-1 = [[-2.0, 1.0], [1.5, -0.5]]
A @ A^-1 = [[1.0, 0.0], [0.0, 1.0]]


In [4]:
import random

inputs = Matrix([[0.5], [0.8], [0.2]])
weights = Matrix([
    [random.uniform(-1, 1) for _ in range(3)]
    for _ in range(2)
])
bias = Matrix([[0.1], [0.1]])

def relu_matrix(m):
    return Matrix([[max(0, val) for val in row] for row in m.data])

pre_activation = weights.matmul(inputs) + bias
output = relu_matrix(pre_activation)

print(f"输入形状: {inputs.shape}")
print(f"权重形状: {weights.shape}")
print(f"输出形状: {output.shape}")
print(f"输出: {output.data}")

输入形状: (3, 1)
权重形状: (2, 3)
输出形状: (2, 1)
输出: [[0.4913818732509474], [0]]


## 这个动手实现的作用是什么？帮助小白理解什么呢？

从零实现 `Matrix` 类的核心作用，是**消除“矩阵运算在黑盒里自动完成”的幻觉**，帮助小白建立对神经网络底层操作的**具体、可触摸的理解**。具体来说，它解决了以下几个常见困惑：

1. **形状为什么必须匹配？**  
   自己写 `matmul` 时，你会亲眼看到循环中的 `for k in range(self.cols)`——只有当第一个矩阵的列数等于第二个矩阵的行数时，这个循环才有意义。形状不匹配会直接导致索引错误，而不是框架抛出的神秘报错。

2. **矩阵乘法 ≠ 逐元素乘法**  
   很多初学者以为 `*` 就是矩阵乘法。通过实现 `element_wise_multiply` 和 `matmul` 两个不同方法，小白会清楚看到：  
   - 逐元素乘法：两个相同形状矩阵的对应位置相乘（`[i][j] * other[i][j]`）。  
   - 矩阵乘法：行与列的点积（对 `k` 求和）。  
   从此不会再混淆。

3. **广播不是“魔法”**  
   当 `matrix + bias` 形状不一致时，框架自动广播。自己实现时，需要显式地复制偏置向量以匹配矩阵的行数——这揭示了广播的本质：**重复利用较小数组的元素**，而不是改变数据本身。

4. **神经网络层就是那三行数学**  
   `output = relu(W @ x + b)` 看起来简单，但如果不亲手用矩阵类计算一遍，很容易把它当成“调用函数”。自己做完 `weights.matmul(inputs) + bias` 再套上 `relu_matrix`，就会明白：  
   - 权重矩阵的每一行对应一个输出神经元。  
   - 矩阵乘法同时为所有输出神经元计算加权和。  
   - 偏置加到每个神经元上。  
   - 激活函数逐元素应用。  

5. **为后续学习框架（NumPy/PyTorch）建立信心**  
   当小白知道“NumPy 的 `@` 只是做了和我一样的循环，只不过用 C 语言加速了”，就不会再把框架当成黑盒。遇到形状错误时，也能自己推理出问题所在。

**一句话总结**：这个动手实现让小白亲手“拆解”了矩阵运算的每一个齿轮，从而真正理解神经网络前向传播的底层数学，而不仅仅是机械地调用 `model.fit()`。